# 08. Heap Sort, Radix Sort and Sorting Applications

The algorithms not covered earlier, plus applications where the **sorting idea** (not just the call) solves the problem.

**Learning order:** HEAP SORT → RADIX SORT → K-SORTED ARRAY → H-INDEX → MAXIMUM GAP → SORT BY PARITY → WHICH SORT WHEN → INTERVIEW THINKING

## 1. Heap sort (in place, O(n log n) guaranteed)

1. **Build a max-heap** inside the array (heapify from the last parent down): O(n).
2. Repeat: swap the root (maximum) with the last element of the heap, shrink the heap, **sift down** the new root.

Time O(n log n) in all cases, space O(1), **not stable**, slower than quicksort in practice (poor cache behaviour). Good when you need the worst-case guarantee **and** O(1) space.

In [1]:
def heap_sort(arr):
    n = len(arr)
    def sift_down(i, size):
        while True:
            largest, l, r = i, 2 * i + 1, 2 * i + 2
            if l < size and arr[l] > arr[largest]: largest = l
            if r < size and arr[r] > arr[largest]: largest = r
            if largest == i:
                return
            arr[i], arr[largest] = arr[largest], arr[i]
            i = largest
    for i in range(n // 2 - 1, -1, -1):          # build the max-heap
        sift_down(i, n)
    for end in range(n - 1, 0, -1):              # move the max to the end, repair the heap
        arr[0], arr[end] = arr[end], arr[0]
        sift_down(0, end)
    return arr

print(heap_sort([4, 10, 3, 5, 1]), heap_sort([2, 1]))   # [1, 3, 4, 5, 10] [1, 2]

[1, 3, 4, 5, 10] [1, 2]


## 2. Radix sort (LSD, non-negative integers)

Sort numbers **digit by digit from the least significant**, each time with a **stable** counting sort on that digit. Stability is essential: it keeps the order from earlier digits. Time O(d · (n + 10)) for `d` digits. Beats n log n when `d` is small.

In [2]:
def radix_sort(nums):
    if not nums:
        return nums
    exp = 1
    while max(nums) // exp > 0:
        buckets = [[] for _ in range(10)]
        for x in nums:
            buckets[(x // exp) % 10].append(x)     # stable: appended in current order
        nums = [x for b in buckets for x in b]
        exp *= 10
    return nums

print(radix_sort([170, 45, 75, 90, 802, 24, 2, 66]))   # [2, 24, 45, 66, 75, 90, 170, 802]

[2, 24, 45, 66, 75, 90, 170, 802]


## 3. Sort a K-sorted array

Every element is at most `k` positions away from its sorted place. A **min-heap of size k + 1** always contains the next smallest element: pop it, push the next. O(n log k).

In [3]:
import heapq

def sort_k_sorted(nums, k):
    heap = nums[:k + 1]
    heapq.heapify(heap)
    res = []
    for x in nums[k + 1:]:
        res.append(heapq.heappushpop(heap, x))
    while heap:
        res.append(heapq.heappop(heap))
    return res

print(sort_k_sorted([6, 5, 3, 2, 8, 10, 9], 3))   # [2, 3, 5, 6, 8, 9, 10]

[2, 3, 5, 6, 8, 9, 10]


## 4. H-Index (counting sort trick)

A researcher has `h` if `h` papers have at least `h` citations each. Citations above `n` can be capped at `n`, so **count** papers per citation value in an array of size `n + 1` and scan from the top. O(n), no sort.

In [4]:
def h_index(citations):
    n = len(citations)
    count = [0] * (n + 1)
    for c in citations:
        count[min(c, n)] += 1
    papers = 0
    for h in range(n, -1, -1):
        papers += count[h]                         # papers with at least h citations
        if papers >= h:
            return h
    return 0

print(h_index([3, 0, 6, 1, 5]), h_index([1, 3, 1]))   # 3 1

3 1


## 5. Maximum Gap (bucket sort, hard)

Maximum difference between **adjacent values in sorted order**, in **O(n)**. With `n` numbers between `lo` and `hi`, the maximum gap is at least `(hi - lo) / (n - 1)`. Use buckets of that size: the answer is never **inside** a bucket, so compare only the **min of a bucket with the max of the previous non-empty bucket**.

In [5]:
def maximum_gap(nums):
    if len(nums) < 2:
        return 0
    lo, hi, n = min(nums), max(nums), len(nums)
    if lo == hi:
        return 0
    size = max(1, (hi - lo) // (n - 1))
    count = (hi - lo) // size + 1
    mins, maxs = [None] * count, [None] * count
    for x in nums:
        i = (x - lo) // size
        mins[i] = x if mins[i] is None else min(mins[i], x)
        maxs[i] = x if maxs[i] is None else max(maxs[i], x)
    best, prev_max = 0, lo
    for i in range(count):
        if mins[i] is None:
            continue
        best = max(best, mins[i] - prev_max)
        prev_max = maxs[i]
    return best

print(maximum_gap([3, 6, 9, 1]), maximum_gap([10]), maximum_gap([1, 10000000]))   # 3 0 9999999

3 0 9999999


## 6. Sort Array By Parity (partition, no full sort)

Move all even numbers before odd numbers. This is one **quicksort-style partition** step with two pointers: O(n), O(1) space. The same idea as the Dutch National Flag.

In [6]:
def sort_by_parity(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        if nums[l] % 2 == 0:
            l += 1
        elif nums[r] % 2 == 1:
            r -= 1
        else:
            nums[l], nums[r] = nums[r], nums[l]
    return nums

out = sort_by_parity([3, 1, 2, 4])
print(all(x % 2 == 0 for x in out[:2]) and all(x % 2 == 1 for x in out[2:]))   # True

True


## 7. Which sort when?

| Situation | Choose |
|---|---|
| general purpose | built-in Timsort |
| small integer range | counting sort |
| fixed-width integers / strings | radix sort |
| uniformly distributed floats | bucket sort |
| guaranteed O(n log n) and O(1) space | heap sort |
| stable + guaranteed + linked list | merge sort |
| k-th element only | quickselect / heap, not a full sort |
| nearly sorted / k-sorted | insertion sort / Timsort / heap of size k + 1 |
| data does not fit in memory | external merge sort |

## 8. Interview thinking

**Say:** "A full sort is O(n log n), but I only need the k largest, so a heap of size k gives O(n log k)." / "Values are bounded by n, so counting sort gives O(n)."

**Common mistakes:** radix sort with an unstable inner sort, negative numbers in radix/counting sort (offset them), heap sort sift-down bounds (`size` not `len(arr)`), assuming heap sort is stable.

## Quick Revision

- Heap sort: build max-heap O(n), swap root with the end, sift down. O(n log n), O(1), unstable.
- Radix: stable counting sort per digit, LSD first.
- K-sorted: heap of size k + 1.
- H-index and maximum gap: counting/bucket tricks give O(n).